In [41]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import csv
import pandas as pd
from datetime import datetime
import pytz 
from fake_useragent import UserAgent
import re

import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [42]:
data = pd.read_csv('final_merged_data.csv')

In [44]:
data.columns

Index(['Geographic Area Name',
       'Estimate!!Percent below poverty level!!UNRELATED INDIVIDUALS FOR WHOM POVERTY STATUS IS DETERMINED!!Did not work',
       'Estimate!!Percent below poverty level!!Population for whom poverty status is determined!!WORK EXPERIENCE!!Population 16 years and over!!Worked full-time, year-round in the past 12 months',
       'Estimate!!Total!!Population for whom poverty status is determined!!AGE!!Under 18 years',
       'Geographic Area Name_x', 'Area_Type', 'Tract',
       ' !!Total:!!Population of one race:!!White alone_PROPORTION',
       ' !!Total:!!Population of one race:!!Black or African American alone_PROPORTION',
       ' !!Total:!!Population of one race:!!American Indian and Alaska Native alone_PROPORTION',
       ' !!Total:!!Population of one race:!!Asian alone_PROPORTION',
       ' !!Total:!!Population of one race:!!Native Hawaiian and Other Pacific Islander alone_PROPORTION',
       ' !!Total:!!Population of one race:!!Some Other Race alone_P

In [45]:
predictors = data[['Census population, 2020','Estimate!!Total!!Population 16 years and over!!AGE!!16 to 19 years','Imprisonment rate per 100,000','Estimate!!Unemployment rate!!Population 16 years and over', ' !!Total:!!Population of one race:!!White alone_PROPORTION',' !!Total:!!Population of one race:!!Black or African American alone_PROPORTION',' !!Total:!!Population of one race:!!American Indian and Alaska Native alone_PROPORTION',' !!Total:!!Population of one race:!!Asian alone_PROPORTION',' !!Total:!!Population of one race:!!Native Hawaiian and Other Pacific Islander alone_PROPORTION',' !!Total:!!Population of one race:!!Some Other Race alone_PROPORTION', 'Area_Type', 'Estimate!!Percent below poverty level!!Population for whom poverty status is determined!!WORK EXPERIENCE!!Population 16 years and over!!Worked full-time, year-round in the past 12 months','Estimate!!Percent below poverty level!!UNRELATED INDIVIDUALS FOR WHOM POVERTY STATUS IS DETERMINED!!Did not work', 'Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years', 'Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Less than high school graduate', 'Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!High school graduate (includes equivalency)', "Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Some college or associate's degree", "Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Bachelor's degree or higher"]].copy()

In [46]:
predictors


,"Census population, 2020",Estimate!!Total!!Population 16 years and over!!AGE!!16 to 19 years,"Imprisonment rate per 100,000",Estimate!!Unemployment rate!!Population 16 years and over,!!Total:!!Population of one race:!!White alone_PROPORTION,!!Total:!!Population of one race:!!Black or African American alone_PROPORTION,!!Total:!!Population of one race:!!American Indian and Alaska Native alone_PROPORTION,!!Total:!!Population of one race:!!Asian alone_PROPORTION,!!Total:!!Population of one race:!!Native Hawaiian and Other Pacific Islander alone_PROPORTION,!!Total:!!Population of one race:!!Some Other Race alone_PROPORTION,Area_Type,"Estimate!!Percent below poverty level!!Population for whom poverty status is determined!!WORK EXPERIENCE!!Population 16 years and over!!Worked full-time, year-round in the past 12 months",Estimate!!Percent below poverty level!!UNRELATED INDIVIDUALS FOR WHOM POVERTY STATUS IS DETERMINED!!Did not work,Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years,Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Less than high school graduate,Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!High school graduate (includes equivalency),Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Some college or associate's degree,Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Bachelor's degree or higher
0,"3,038",65,99,1.0,0.615537,0.047400,0.000658,0.177090,0.003621,0.026333,Urban,2.1,21.6,1644,24,18,141,1461
1,"2,001",22,50,7.9,0.710145,0.019990,0.001499,0.103448,0.001999,0.020490,Urban,1.9,52.9,1174,18,46,95,1015
2,"5,504",202,18,3.4,0.620094,0.101926,0.004724,0.100654,0.004906,0.034702,Urban,0.0,37.9,3193,82,87,463,2561
3,"4,112",99,146,2.5,0.672422,0.070282,0.002918,0.097033,0.001946,0.029669,Urban,0.5,39.6,2705,78,165,362,2100
4,"3,644",111,410,6.4,0.537047,0.171515,0.006586,0.084248,0.001372,0.046103,Urban,1.2,66.0,2409,30,293,294,1792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7352,"2,738",228,219,1.6,0.339664,0.011322,0.016801,0.167275,0.007670,0.254565,Urban,5.9,0.0,1370,164,247,616,343
7353,"1,653",35,241,0.0,0.410163,0.017544,0.008469,0.128252,0.006050,0.196007,Urban,0.0,72.0,837,10,198,195,434
7354,"1,361",102,293,5.0,0.396032,0.005878,0.047024,0.017634,0.001470,0.351212,Urban,0.0,93.1,877,152,177,438,110
7355,"3,600",162,744,9.2,0.465000,0.033333,0.021944,0.134722,0.004444,0.175833,Urban,8.5,68.3,2398,382,690,892,434


In [58]:
predictors = predictors.rename(columns={'Census population, 2020': 'population'})
predictors = predictors.rename(columns={'Estimate!!Unemployment rate!!Population 16 years and over': 'unemployment_rate'})
predictors = predictors.rename(columns={'Imprisonment rate per 100,000': 'incarceration_rate'})
predictors = predictors.rename(columns={'Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years': 'education_total'})
predictors = predictors.rename(columns={'Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Less than high school graduate': 'education_less_than_highschool'})
predictors = predictors.rename(columns={'Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!High school graduate (includes equivalency)': 'education_highschool'})
predictors = predictors.rename(columns={"Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Some college or associate's degree": 'education_some_college'})
predictors = predictors.rename(columns={"Estimate!!Total!!EDUCATIONAL ATTAINMENT!!Population 25 to 64 years!!Bachelor's degree or higher": 'education_bachelors_or_higher'})
predictors = predictors.rename(columns={"Estimate!!Total!!Population 16 years and over!!AGE!!16 to 19 years": 'total_population_16_to_19'})


predictors = predictors.rename(columns={" !!Total:!!Population of one race:!!White alone_PROPORTION": 'white_proportion'})
predictors = predictors.rename(columns={" !!Total:!!Population of one race:!!Black or African American alone_PROPORTION": 'black_proportion'})
predictors = predictors.rename(columns={" !!Total:!!Population of one race:!!American Indian and Alaska Native alone_PROPORTION": 'native_american_proportion'})
predictors = predictors.rename(columns={" !!Total:!!Population of one race:!!Asian alone_PROPORTION": 'asian_proportion'})
predictors = predictors.rename(columns={" !!Total:!!Population of one race:!!Native Hawaiian and Other Pacific Islander alone_PROPORTION": 'native_hawaiian_pac_islander_proportion'})
predictors = predictors.rename(columns={"Estimate!!Percent below poverty level!!Population for whom poverty status is determined!!WORK EXPERIENCE!!Population 16 years and over!!Worked full-time, year-round in the past 12 months": 'percent_poverty_worked'})
predictors = predictors.rename(columns={"Estimate!!Percent below poverty level!!UNRELATED INDIVIDUALS FOR WHOM POVERTY STATUS IS DETERMINED!!Did not work": 'percent_poverty_did_not_work'})




In [59]:
predictors = predictors.replace('-', pd.NA)
predictors.dropna(inplace=True)

In [60]:
# # # Remove commas from the 'incarceration_rate' column
# predictors['incarceration_rate'] = predictors['incarceration_rate'].str.replace(',', '')
# predictors['unemployment_rate'] = predictors['unemployment_rate'].str.replace(',', '')
# predictors['population'] = predictors['population'].str.replace(',', '')


# # # Convert the column to appropriate type
# predictors['incarceration_rate'] = predictors['incarceration_rate'].astype(float)
# predictors['unemployment_rate'] = predictors['unemployment_rate'].astype(float)
# predictors['population'] = predictors['population'].astype(float)
predictors['percent_poverty_worked'] = predictors['percent_poverty_worked'].astype(float)
predictors['percent_poverty_did_not_work'] = predictors['percent_poverty_did_not_work'].astype(float)


In [61]:
predictors.dtypes

population                                 float64
total_population_16_to_19                    int64
incarceration_rate                         float64
unemployment_rate                          float64
white_proportion                           float64
black_proportion                           float64
native_american_proportion                 float64
asian_proportion                           float64
native_hawaiian_pac_islander_proportion    float64
other_race_proportion                      float64
Area_Type                                   object
percent_poverty_worked                     float64
percent_poverty_did_not_work               float64
education_total                              int64
education_less_than_highschool               int64
education_highschool                         int64
education_some_college                       int64
education_bachelors_or_higher                int64
dtype: object

In [64]:
reg = smf.ols('incarceration_rate ~ unemployment_rate + C(Area_Type) + population + percent_poverty_worked + percent_poverty_did_not_work + white_proportion + black_proportion + native_american_proportion + asian_proportion + native_hawaiian_pac_islander_proportion + other_race_proportion  + education_total + education_less_than_highschool + education_highschool + education_some_college + education_bachelors_or_higher + total_population_16_to_19', data = predictors).fit()
reg.summary() 

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     incarceration_rate   R-squared:                       0.120
Model:                            OLS   Adj. R-squared:                  0.118
Method:                 Least Squares   F-statistic:                     62.01
Date:                Tue, 03 Oct 2023   Prob (F-statistic):          6.84e-188
Time:                        15:32:38   Log-Likelihood:                -58936.
No. Observations:                7281   AIC:                         1.179e+05
Df Residuals:                    7264   BIC:                         1.180e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
===========================================================================================================
                                              coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept                                6719.9333    256.577     26.191      0.000    6216.968    7222.898
C(Area_Type)[T.Urban]                    -126.0418     48.827     -2.581      0.010    -221.758     -30.326
unemployment_rate                          13.9628      2.562      5.449      0.000       8.940      18.986
population                                 -0.0115      0.014     -0.846      0.398      -0.038       0.015
percent_poverty_worked                      9.8368      2.825      3.482      0.001       4.298      15.375
percent_poverty_did_not_work                1.0565      0.466      2.269      0.023       0.144       1.969
white_proportion                        -7257.5358    278.720    -26.039      0.000   -7803.909   -6711.163
black_proportion                        -5775.0604    275.378    -20.971      0.000   -6314.882   -5235.239
native_american_proportion              -7166.4937    669.301    -10.707      0.000   -8478.518   -5854.469
asian_proportion                        -6865.4870    265.911    -25.819      0.000   -7386.749   -6344.225
native_hawaiian_pac_islander_proportion -9645.9791   1690.824     -5.705      0.000    -1.3e+04   -6331.472
other_race_proportion                   -8292.4492    343.222    -24.161      0.000   -8965.263   -7619.635
education_total                         -8.166e-05      0.019     -0.004      0.997      -0.037       0.037
education_less_than_highschool              0.2987      0.046      6.504      0.000       0.209       0.389
education_highschool                       -0.0573      0.049     -1.168      0.243      -0.153       0.039
education_some_college                     -0.2174      0.039     -5.504      0.000      -0.295      -0.140
education_bachelors_or_higher              -0.0241      0.019     -1.245      0.213      -0.062       0.014
total_population_16_to_19                  -0.0609      0.058     -1.056      0.291      -0.174       0.052
==============================================================================
Omnibus:                    25712.233   Durbin-Watson:                   1.883
Prob(Omnibus):                  0.000   Jarque-Bera (JB):       9164229900.530
Skew:                          69.090   Prob(JB):                         0.00
Kurtosis:                    5497.407   Cond. No.                     2.81e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 2.77e-22. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

- Use GVIF instead of VIF

In [69]:
reg = smf.ols('incarceration_rate ~ unemployment_rate + C(Area_Type) + population + percent_poverty_worked + black_proportion + white_proportion + asian_proportion + percent_poverty_did_not_work  + education_total + education_less_than_highschool + education_highschool + education_some_college + education_bachelors_or_higher + total_population_16_to_19', data = predictors).fit()
reg.summary() 

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     incarceration_rate   R-squared:                       0.048
Model:                            OLS   Adj. R-squared:                  0.047
Method:                 Least Squares   F-statistic:                     28.46
Date:                Tue, 03 Oct 2023   Prob (F-statistic):           3.53e-69
Time:                        15:36:53   Log-Likelihood:                -59221.
No. Observations:                7281   AIC:                         1.185e+05
Df Residuals:                    7267   BIC:                         1.186e+05
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
==================================================================================================
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                        868.0229     91.582      9.478      0.000     688.495    1047.551
C(Area_Type)[T.Urban]            -49.5062     49.425     -1.002      0.317    -146.393      47.381
unemployment_rate                 16.2054      2.661      6.089      0.000      10.989      21.422
population                        -0.0188      0.014     -1.337      0.181      -0.046       0.009
percent_poverty_worked             2.2180      2.915      0.761      0.447      -3.496       7.932
black_proportion                 -43.5802    143.486     -0.304      0.761    -324.854     237.694
white_proportion                -820.7197     90.447     -9.074      0.000    -998.022    -643.417
asian_proportion                -817.6429     94.899     -8.616      0.000   -1003.673    -631.613
percent_poverty_did_not_work      -0.0635      0.481     -0.132      0.895      -1.007       0.880
education_total                   -0.0292      0.020     -1.487      0.137      -0.068       0.009
education_less_than_highschool    -0.1069      0.044     -2.419      0.016      -0.193      -0.020
education_highschool              -0.0080      0.051     -0.157      0.875      -0.108       0.092
education_some_college             0.0476      0.039      1.215      0.225      -0.029       0.124
education_bachelors_or_higher      0.0381      0.020      1.924      0.054      -0.001       0.077
total_population_16_to_19         -0.0409      0.060     -0.682      0.495      -0.158       0.077
==============================================================================
Omnibus:                    26972.464   Durbin-Watson:                   1.956
Prob(Omnibus):                  0.000   Jarque-Bera (JB):      13467036716.606
Skew:                          79.849   Prob(JB):                         0.00
Kurtosis:                    6663.722   Cond. No.                     4.54e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 1.07e-22. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""